# GANForge -- AC-GAN (Standalone Notebook)

**Conditional generation + auxiliary classification**

This is a **self-contained extract** of the AC-GAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="3"></a>
## 3. AC-GAN -- Conditional generation + auxiliary classification

### 3.1 Problem Statement & Why this GAN

Odena et al. (2017) noticed that cGAN's discriminator only ever answers
"real or fake, *given* the label" -- it never has to *predict* the label
itself, so the gradient signal for "did you draw the right class" is
weak. AC-GAN's discriminator instead outputs **two heads**: real/fake
*and* a full class-probability distribution (an auxiliary classifier). The
generator is now pushed by two error signals -- fool the discriminator,
*and* be classified as the requested class -- which empirically yields
sharper, more class-faithful samples on harder, natural-image datasets.

We move to CIFAR-10 (real color photos, harder than MNIST/FashionMNIST) to
show AC-GAN's improvement matters most when the conditioning problem is
genuinely difficult.

### 3.2 Data Collection & Preparation

In [ ]:
# We use a 4-class CIFAR-10 subset (airplane, automobile, bird, cat) purely
# for training-time budget -- the architecture below scales to all 10
# classes unchanged if you set ACGAN_CLASS_IDS = list(range(10)).
ACGAN_CLASS_IDS = [0, 1, 2, 3]
ACGAN_CLASS_NAMES = ["airplane", "automobile", "bird", "cat"]
ACGAN_IMG_SIZE = 32
ACGAN_BATCH_SIZE = 128
ACGAN_NZ = 100
ACGAN_EPOCHS = 8

acgan_transform = T.Compose([T.ToTensor(), T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

try:
    full_cifar = torchvision.datasets.CIFAR10(root="data", train=True, download=True, transform=acgan_transform)
    idx = [i for i, (_, y) in enumerate(full_cifar) if y in ACGAN_CLASS_IDS]
    acgan_dataset = torch.utils.data.Subset(full_cifar, idx)
    # remap original CIFAR labels {0,1,2,3} -> contiguous {0..3} (already contiguous here, but keep for generality)
    label_map = {orig: new for new, orig in enumerate(ACGAN_CLASS_IDS)}
    print(f"Loaded real CIFAR-10 subset: {len(acgan_dataset)} images across {len(ACGAN_CLASS_IDS)} classes")
except Exception as e:
    print(f"CIFAR-10 download failed ({e}); falling back to synthetic colored-shape classes.")
    rng = np.random.default_rng(SEED)
    n_per_class, imgs, labels = 300, [], []
    for c in range(len(ACGAN_CLASS_IDS)):
        color = torch.tensor([[1., -1, -1], [-1, 1, -1], [-1, -1, 1], [1, 1, -1]][c])
        for _ in range(n_per_class):
            img = torch.randn(3, ACGAN_IMG_SIZE, ACGAN_IMG_SIZE) * 0.05 + color.view(3, 1, 1)
            imgs.append(img.clamp(-1, 1)); labels.append(c)
    acgan_dataset = TensorDataset(torch.stack(imgs), torch.tensor(labels))
    label_map = {c: c for c in range(len(ACGAN_CLASS_IDS))}

class RemapLabels(Dataset):
    '''Thin wrapper: remap arbitrary source labels to contiguous 0..K-1 ids the Embedding layers expect.'''
    def __init__(self, base, label_map):
        self.base, self.label_map = base, label_map
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return x, self.label_map[int(y)]

acgan_dataset = RemapLabels(acgan_dataset, label_map)
acgan_loader = DataLoader(acgan_dataset, batch_size=ACGAN_BATCH_SIZE, shuffle=True, drop_last=True)
real_batch, real_labels = next(iter(acgan_loader))
show_images(real_batch[:64], title="Real AC-GAN training samples", nrow=8)


### 3.3 Model Architecture

In [ ]:
ACGAN_N_CLASSES = len(ACGAN_CLASS_IDS)

class ACGAN_Generator(nn.Module):
    '''Same conditioning-by-concatenation idea as cGAN, scaled up with one
    extra upsampling stage (32x32 RGB instead of grayscale) and more
    channels since natural images carry far more information than digits.'''
    def __init__(self, nz=ACGAN_NZ, n_classes=ACGAN_N_CLASSES, ngf=64, nc=3, embed_dim=50):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, embed_dim)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + embed_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, y):
        y_emb = self.label_embed(y)
        zy = torch.cat([z, y_emb], dim=1)
        return self.net(zy.view(zy.size(0), -1, 1, 1))


class ACGAN_Discriminator(nn.Module):
    '''Shared convolutional trunk -> two heads:
    - `adv_head`: real/fake logit (unconditional -- unlike cGAN, AC-GAN's
      discriminator is NOT told the label up front; it has to recover it).
    - `cls_head`: class logits over ACGAN_N_CLASSES, trained with
      cross-entropy on BOTH real images (their true label) and fake images
      (the label G was asked to draw) -- this is what forces G to be
      class-faithful, not just realistic.'''
    def __init__(self, n_classes=ACGAN_N_CLASSES, nc=3, ndf=64):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
        )
        self.adv_head = nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False)
        self.cls_head = nn.Conv2d(ndf * 4, n_classes, 4, 1, 0, bias=False)

    def forward(self, x):
        feat = self.trunk(x)
        return self.adv_head(feat).view(-1), self.cls_head(feat).view(feat.size(0), -1)


G_acgan = ACGAN_Generator().to(DEVICE).apply(weights_init)
D_acgan = ACGAN_Discriminator().to(DEVICE).apply(weights_init)


### 3.4 Loss Functions & Optimizers

In [ ]:
acgan_adv_criterion = nn.BCEWithLogitsLoss()   # real vs fake
acgan_cls_criterion = nn.CrossEntropyLoss()    # which class
acgan_opt_G = torch.optim.Adam(G_acgan.parameters(), lr=2e-4, betas=(0.5, 0.999))
acgan_opt_D = torch.optim.Adam(D_acgan.parameters(), lr=2e-4, betas=(0.5, 0.999))
acgan_fixed_noise = torch.randn(ACGAN_N_CLASSES * 8, ACGAN_NZ, device=DEVICE)
acgan_fixed_labels = torch.arange(ACGAN_N_CLASSES, device=DEVICE).repeat_interleave(8)


### 3.5 Training Loop

In [ ]:
acgan_history = []
for epoch in range(ACGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter(), "acc": AvgMeter()}
    pbar = tqdm(acgan_loader, desc=f"[AC-GAN] epoch {epoch+1}/{ACGAN_EPOCHS}")
    for real, y in pbar:
        real, y = real.to(DEVICE), y.to(DEVICE)
        b = real.size(0)

        # ---- Discriminator: adversarial loss (real vs fake) + classification loss (on REAL images only here) ----
        D_acgan.zero_grad(set_to_none=True)
        adv_real, cls_real = D_acgan(real)
        loss_d_adv_real = acgan_adv_criterion(adv_real, torch.ones(b, device=DEVICE))
        loss_d_cls_real = acgan_cls_criterion(cls_real, y)

        z = torch.randn(b, ACGAN_NZ, device=DEVICE)
        fake_y = torch.randint(0, ACGAN_N_CLASSES, (b,), device=DEVICE)
        fake = G_acgan(z, fake_y)
        adv_fake, cls_fake = D_acgan(fake.detach())
        loss_d_adv_fake = acgan_adv_criterion(adv_fake, torch.zeros(b, device=DEVICE))
        # Classification loss on fakes too: teaches D to recognize class-defining
        # features regardless of real/fake, which in turn sharpens the
        # gradient G receives about "did I draw the requested class".
        loss_d_cls_fake = acgan_cls_criterion(cls_fake, fake_y)

        loss_d = loss_d_adv_real + loss_d_adv_fake + loss_d_cls_real + loss_d_cls_fake
        loss_d.backward()
        acgan_opt_D.step()

        # ---- Generator: fool D AND be classified as the requested class ----
        G_acgan.zero_grad(set_to_none=True)
        adv_fake_g, cls_fake_g = D_acgan(fake)
        loss_g_adv = acgan_adv_criterion(adv_fake_g, torch.ones(b, device=DEVICE))
        loss_g_cls = acgan_cls_criterion(cls_fake_g, fake_y)
        loss_g = loss_g_adv + loss_g_cls
        loss_g.backward()
        acgan_opt_G.step()

        acc = (cls_real.argmax(1) == y).float().mean().item()
        meters["D"].update(loss_d.item()); meters["G"].update(loss_g.item()); meters["acc"].update(acc)
        pbar.set_postfix(loss_D=f"{loss_d.item():.3f}", loss_G=f"{loss_g.item():.3f}", D_cls_acc=f"{acc:.2f}")

    acgan_history.append((meters["G"].avg, meters["D"].avg, meters["acc"].avg))
    with torch.no_grad():
        G_acgan.eval()
        sample = G_acgan(acgan_fixed_noise, acgan_fixed_labels)
        G_acgan.train()
    show_images(sample, title=f"AC-GAN samples after epoch {epoch+1} (rows=classes)", nrow=8)
    save_checkpoint({"G": G_acgan.state_dict(), "D": D_acgan.state_dict()}, f"{CHECKPOINT_DIR}/acgan_epoch{epoch+1}.pt")

print("AC-GAN training complete.")


### 3.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def acgan_generate(class_id: int, n_samples: int = 8, seed: int = None) -> torch.Tensor:
    if seed is not None:
        torch.manual_seed(seed)
    G_acgan.eval()
    z = torch.randn(n_samples, ACGAN_NZ, device=DEVICE)
    y = torch.full((n_samples,), class_id, dtype=torch.long, device=DEVICE)
    imgs = G_acgan(z, y)
    G_acgan.train()
    return imgs

show_images(acgan_generate(2, 8), title=f"AC-GAN inference: class='{ACGAN_CLASS_NAMES[2]}'", nrow=8, figsize=(10, 2))


### 3.7 Visualization & Results

In [ ]:
gs, ds, accs = zip(*acgan_history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(gs, label="G"); axes[0].plot(ds, label="D"); axes[0].legend(); axes[0].set_title("Losses")
axes[1].plot(accs); axes[1].set_title("D's auxiliary classifier accuracy on real images")
plt.show()

# Quantitative sanity check: does the AUXILIARY CLASSIFIER agree that
# generated images match the class they were conditioned on? This is a
# cheap proxy for the "Inception Score"-style self-consistency metrics
# used in the AC-GAN paper.
with torch.no_grad():
    D_acgan.eval()
    z = torch.randn(200, ACGAN_NZ, device=DEVICE)
    y_req = torch.randint(0, ACGAN_N_CLASSES, (200,), device=DEVICE)
    fakes = G_acgan(z, y_req)
    _, cls_logits = D_acgan(fakes)
    agreement = (cls_logits.argmax(1) == y_req).float().mean().item()
    D_acgan.train()
print(f"Self-consistency (D's classifier agrees with requested class): {agreement*100:.1f}%")


### 3.8 User Input → Output Demo

In [ ]:
USER_REQUESTED_CLASS = "cat"
class_idx = ACGAN_CLASS_NAMES.index(USER_REQUESTED_CLASS)
show_images(acgan_generate(class_idx, n_samples=16), title=f"User requested: '{USER_REQUESTED_CLASS}'", nrow=4)


### 3.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Training the auxiliary classifier on *both* real and fake
  images (not just real, as some simplified tutorials do) is what actually
  gives the generator a useful classification gradient -- classifying only
  reals gives G no direct feedback about its own class-faithfulness.
- ⚠️ **Warning.** AC-GAN is known to have a subtle failure mode: with many
  classes, the auxiliary loss can dominate and push the generator toward
  **low-diversity, highly class-separable but less realistic** samples
  (the discriminator's easiest way to raise classification accuracy is
  often to make classes maximally different-looking, not maximally
  photorealistic).

**This GAN solved the problem of: class-conditional generation with an
explicit, learnable notion of "class identity" baked into the
discriminator -- improving class-faithfulness over plain cGAN on harder,
natural-image data.**